# 🧬 SimDock Polymer v2.0 - Cloud GPU Environment
Run this notebook step-by-step to execute the *real* physics simulation (GNINA docking + OpenMM 10ns MD) on a free Colab GPU.

In [ ]:
# 1. Setup Conda (The kernel will automatically restart after this finishes. That is normal!)
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# 2. Install heavy chemistry dependencies via Conda
import condacolab
condacolab.check()
!conda install -y -c conda-forge openmm mdtraj rdkit biopython openmmforcefields openff-toolkit

In [ ]:
# 3. Install Python pip packages & System Binaries
!pip install meeko gemmi pdbfixer fastapi uvicorn pydantic matplotlib pyyaml pandas scipy numpy

!wget -q https://github.com/gnina/gnina/releases/download/v1.0.3/gnina -O /usr/bin/gnina
!chmod +x /usr/bin/gnina
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /usr/bin/vina
!chmod +x /usr/bin/vina
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/bin/cloudflared
!chmod +x /usr/bin/cloudflared

In [ ]:
# 4. Retrieve project source files
# Clones from your GitHub repository directly. If cloning fails (e.g. private repo restrictions),
# it falls back to unzipping 'SimDock_Project.zip' if you uploaded it manually.
!git clone https://github.com/messiay/PolymerDock.git polymerDock || (unzip -q -o SimDock_Project.zip -d polymerDock && echo 'Fallback to local SimDock_Project.zip successful')
%cd polymerDock

In [ ]:
# 5. Launch the SimDock Web Application with Cloudflare Tunnel!
import sys
import subprocess
import time
import re
import os

print("Starting FastAPI Uvicorn Server in Conda-Colab environment...")
# Launch backend+frontend on port 8501 using sys.executable
subprocess.Popen([sys.executable, "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8501"])
time.sleep(2)

print("Starting Cloudflare Tunnel...")
log_path = "cloudflare.log"
with open(log_path, "w") as f:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=f, stderr=f)

# Poll the log file for the public tunnel URL
tunnel_url = None
print("Retrieving public HTTPS URL from Cloudflare...")
for _ in range(12):
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n✅ SUCCESS! Click the link below to open your app in a new tab:")
    print(f"\n👉 {tunnel_url} 👈\n")
else:
    print("\n⚠️ Could not automatically detect Cloudflare URL.")
    print("Check cloudflare.log for details, or try the secure port window fallback:")
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(8501)
    except Exception:
        pass